## Loading Processor

In [2]:
from pathlib import Path

BASE_DIR = Path.cwd().parent

from transformers import LayoutLMv3Processor

# processor = LayoutLMv3Processor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)
# processor.save_pretrained("receipt_model")

MODEL_PATH = BASE_DIR / "artifacts" / "receipt_model"

processor = LayoutLMv3Processor.from_pretrained(MODEL_PATH)

In [3]:
import pytesseract

def extract_words_boxes(image):
    data = pytesseract.image_to_data(image, output_type=pytesseract.Output.DICT)

    words = []
    boxes = []

    h, w = image.size[1], image.size[0]

    for i, text in enumerate(data["text"]):
        text = text.strip()
        if not text:
            continue

        conf = int(data["conf"][i])
        if conf < 40:
            continue

        x, y, bw, bh = (
            data["left"][i],
            data["top"][i],
            data["width"][i],
            data["height"][i]
        )

        # normalize 0–1000
        box = [
            int(1000 * x / w),
            int(1000 * y / h),
            int(1000 * (x + bw) / w),
            int(1000 * (y + bh) / h),
        ]

        words.append(text)
        boxes.append(box)

    return words, boxes

In [4]:
# Fuzzy matching label assignment¶

def fuzzy_match(a, b, threshold=85):
    return fuzz.ratio(a.lower(), b.lower()) >= threshold


def assign_labels(words, ground_truth):
    labels = ["O"] * len(words)
    texts = [w for w in words]

    for field, value in ground_truth.items():
        if not value:
            continue

        value_tokens = value.split()
        n = len(value_tokens)

        for i in range(len(texts) - n + 1):
            window = texts[i:i+n]
            window_text = " ".join(window)

            if fuzzy_match(window_text, value):
                labels[i] = f"B-{field.upper()}"
                for j in range(1, n):
                    labels[i+j] = f"I-{field.upper()}"

    return labels

## Processing the Training data

In [5]:
TRAIN_PATH = BASE_DIR / "receipts" / "train"

TRAIN_IMG_PATH = TRAIN_PATH / "img"
TRAIN_LABEL_PATH = TRAIN_PATH / "entities"

train_images = list(TRAIN_IMG_PATH.glob("*.jpg"))

# Loading the true labels
import json

images = list(TRAIN_IMG_PATH.glob("*.jpg"))
sample_image_path = images[0]

label_file = TRAIN_LABEL_PATH / f"{sample_image_path.stem}.txt"

with open(label_file) as f:
    ground_truth = json.load(f)

ground_truth

{'company': 'BOOK TA .K (TAMAN DAYA) SDN BHD',
 'date': '25/12/2018',
 'address': 'NO.53 55,57 & 59, JALAN SAGU 18, TAMAN DAYA, 81100 JOHOR BAHRU, JOHOR.',
 'total': '9.00'}

In [6]:
label_list = [
    "O",
    "B-COMPANY", "I-COMPANY",
    "B-DATE", "I-DATE",
    "B-TOTAL", "I-TOTAL",
    "B-ADDRESS", "I-ADDRESS"
]

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

In [7]:
from tqdm import tqdm
from PIL import Image
from fuzzywuzzy import fuzz
from datasets import Dataset

train_data = []

for img_path in tqdm(train_images):
    image = Image.open(img_path).convert("RGB")

    words, boxes = extract_words_boxes(image)

    with open(TRAIN_LABEL_PATH / f"{img_path.stem}.txt") as f:
        gt = json.load(f)

    labels = assign_labels(words, gt)

    encoding = processor(
        image,
        words,
        boxes=boxes,
        word_labels=[label2id[l] for l in labels],
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    train_data.append({
        k: v.squeeze() for k, v in encoding.items()
    })

train_dataset = Dataset.from_list(train_data)
train_dataset.set_format(type="torch")

100%|████████████████████████████████████████████████████████████████████████████████| 681/681 [30:51<00:00,  2.72s/it]


### Processing the test dataset

In [9]:
TEST_PATH = BASE_DIR / "receipts" / "test"

TEST_IMG_PATH = TEST_PATH / "img"
TEST_LABEL_PATH = TEST_PATH / "entities"

test_images = list(TEST_IMG_PATH.glob("*.jpg"))


test_data = []

for img_path in tqdm(test_images):
    image = Image.open(img_path).convert("RGB")

    words, boxes = extract_words_boxes(image)

    with open(TEST_LABEL_PATH / f"{img_path.stem}.txt") as f:
        gt = json.load(f)

    labels = assign_labels(words, gt)

    encoding = processor(
        image,
        words,
        boxes=boxes,
        word_labels=[label2id[l] for l in labels],
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    test_data.append({
        k: v.squeeze() for k, v in encoding.items()
    })

test_dataset = Dataset.from_list(test_data)
test_dataset.set_format(type="torch")

100%|████████████████████████████████████████████████████████████████████████████████| 146/146 [06:22<00:00,  2.62s/it]


In [10]:
print(train_dataset[0].keys())
print(test_dataset[0].keys())

dict_keys(['input_ids', 'attention_mask', 'bbox', 'labels', 'pixel_values'])
dict_keys(['input_ids', 'attention_mask', 'bbox', 'labels', 'pixel_values'])


In [11]:
ENCODED_TRAIN_DATASET = BASE_DIR / "artifacts"/ "encoded_train_receipts_dataset"
ENCODED_TEST_DATASET = BASE_DIR / "artifacts"/ "encoded_test_receipts_dataset"

train_dataset.save_to_disk(ENCODED_TRAIN_DATASET)
test_dataset.save_to_disk(ENCODED_TEST_DATASET)

Saving the dataset (0/1 shards):   0%|          | 0/681 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/146 [00:00<?, ? examples/s]

In [ ]:
# loading the dataset

# from datasets import load_from_disk

# dataset = load_from_disk(ENCODED_TRAIN_DATASET)
# dataset.set_format(type="torch")

## Model training and evaluation

In [12]:
from transformers import LayoutLMv3Processor, LayoutLMv3ForTokenClassification, Trainer, TrainingArguments
from sklearn.metrics import classification_report

In [19]:
import numpy as np

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=2)
    labels = p.label_ids

    true_preds = []
    true_labels = []

    for pred, lab in zip(preds, labels):
        for p_i, l_i in zip(pred, lab):
            if l_i != -100:
                true_preds.append(id2label[p_i])
                true_labels.append(id2label[l_i])

    report = classification_report(
        true_labels,
        true_preds,
        output_dict=True,
        zero_division=0
    )

    return {
        "precision": report["weighted avg"]["precision"],
        "recall": report["weighted avg"]["recall"],
        "f1": report["weighted avg"]["f1-score"]
    }

In [14]:
model = LayoutLMv3ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

Some weights of LayoutLMv3ForTokenClassification were not initialized from the model checkpoint at microsoft/layoutlmv3-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [17]:
training_args = TrainingArguments(
    output_dir="./layoutlmv3_receipts",
    per_device_train_batch_size=2,
    num_train_epochs=2,
    logging_steps=100,
    save_strategy="no"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

2026/03/28 14:38:24 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/03/28 14:38:24 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/03/28 14:38:24 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/03/28 14:38:24 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/03/28 14:38:24 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/03/28 14:38:24 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/03/28 14:38:25 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/28 14:38:25 INFO mlflow.store.db.utils: Updating database tables
2026/03/28 14:38:25 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/03/28 14:38:25 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/03/28 14:38:26 INFO alembic.runtime.migration: Running upgrade  -> 451aebb31d03, add metric step
2026/03/28 14:3

Step,Training Loss
100,0.399400
200,0.194600
300,0.164700
400,0.137300
500,0.121800
600,0.129800


TrainOutput(global_step=682, training_loss=0.1792275877642142, metrics={'train_runtime': 11901.0249, 'train_samples_per_second': 0.114, 'train_steps_per_second': 0.057, 'total_flos': 359036912572416.0, 'train_loss': 0.1792275877642142, 'epoch': 2.0})

In [20]:
metrics = trainer.evaluate()
print(metrics)

{'eval_loss': 0.13320286571979523, 'eval_precision': 0.9610338241076253, 'eval_recall': 0.9563467703430871, 'eval_f1': 0.9578333985448879, 'eval_runtime': 352.0611, 'eval_samples_per_second': 0.415, 'eval_steps_per_second': 0.054, 'epoch': 2.0}


In [23]:
import pandas as pd

metrics_df = pd.DataFrame([metrics])
metrics_df

,eval_loss,eval_precision,eval_recall,eval_f1,eval_runtime,eval_samples_per_second,eval_steps_per_second,epoch
0,0.133203,0.961034,0.956347,0.957833,352.0611,0.415,0.054,2.0


In [ ]:
MODEL_PATH = BASE_DIR / "artifacts" / "receipt_model"

trainer.save_model(MODEL_PATH)